In [7]:
import os
import sys
project_root = "/ssd2/letitiaz/cp_project/code/open-clip/src"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import open_clip
import torch
from test_utiles import *
from unseen_utiles import *

pretrained_ckpt_path = "/ssd2/letitiaz/cp_project/data/logs/ViT-B-16-experiment8/2025_09_03-06_18_00-model_ViT-B-16-lr_0.001-b_175-j_4-p_fp32/checkpoints/epoch_35.pt"
device = torch.device("cuda:7" if torch.cuda.is_available() else "cpu")

tokenizer = open_clip.get_tokenizer(
    model_name="ViT-B-16",
    context_length=256,
    tokenizer_type="hf:gpt2",  
    additional_special_tokens=['<CONC_TOKEN>', '<TIME_TOKEN>', '<COMPOUND_TOKEN>']
)

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    model_name="ViT-B-16",
    pretrained= pretrained_ckpt_path,
    precision="fp32",
    device=device,
    tokenizer=tokenizer,
    output_dict=True,
    load_weights_only=True,
    special_tokens=['<CONC_TOKEN>', '<TIME_TOKEN>', '<COMPOUND_TOKEN>'],
    use_enhanced_clip=True  
)
model.eval()
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Context length:", context_length)
print("Vocab size:", vocab_size)


get_tokenizer called with model_name: ViT-B-16, context_length: 256, tokenizer_type: hf:gpt2
Added 3 special tokens to gpt2 tokenizer
   <CONC_TOKEN> -> ID: 50257
   <TIME_TOKEN> -> ID: 50258
   <COMPOUND_TOKEN> -> ID: 50259
Set pad_token to eos_token for GPT tokenizer
[Auto] Fetched special token IDs from tokenizer: {'<CONC_TOKEN>': 50257, '<TIME_TOKEN>': 50258, '<COMPOUND_TOKEN>': 50259}
Model parameters: 151,468,033
Context length: 256
Vocab size: 50260


In [9]:
compound_moa_list = [
    ("esomeprazole", "potassium–transporting atpase inhibitor"),
    ("regorafenib", "nerve growth factor receptor trk–a"),
    ("sacubitril", "neprilysin inhibitor"),
    ("buparlisib", "pi3–kinase class i inhibitor"),
    ("dexamethasone", "glucocorticoid receptor agonist"),
    ("nimodipine", "voltage–gated l–type calcium channel blocker"),
    ("az258", "aurora kinase inhibitors"),
    ("colchicine", "tubulin inhibitor"),
    ("mg-132", "protein degradation"),
    ("fluphenazine", "dopamine d2 receptor antagonist"),
    ("dapoxetine", "serotonin transporter inhibitor"),
    ("nilotinib", "bcr/abl fusion protein"),
    ("oxybuprocaine", "sodium channel alpha subunit blocker"),
]

compound_moa_list = [(c, m.replace("–", "-")) for c, m in compound_moa_list]
jsonl_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl_moa/final/test_1_cleaned.jsonl'
npz_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl/metadata/all_compounds_embedding.npz'
image_base_dir = "/ssd2/letitiaz/cp_project/data"
for compound, moa in compound_moa_list:
    print(f"\nRunning matcher for:")
    print(f"   Compound: {compound}")
    print(f"   MoA     : {moa}")
    print("=" * 60)

    matcher = CompoundMatcher(
        jsonl_path=jsonl_path,
        npz_path=npz_path,
        image_base_dir=image_base_dir,
        tokenizer=tokenizer,
        model=model,
        device=device,
        target_compound=compound,
        target_moa=moa,
        num_samples=100,
        similarity_threshold=0.4,  
        batch_size=16
    )

    matcher.run_all()



Running matcher for:
   Compound: esomeprazole
   MoA     : potassium-transporting atpase inhibitor
Found 100 samples for compound 'esomeprazole' (sim ≥ 0.4)


Loading batches:   0%|          | 0/7 [00:00<?, ?it/s]

Encoding batches: 100%|██████████| 7/7 [00:01<00:00,  6.10it/s]


Feature extraction complete.
Text features shape: torch.Size([100, 512])
Image features shape: torch.Size([100, 512])
Cosine Similarity Mean: 0.2008
Cosine Similarity Std:  0.1097

Running matcher for:
   Compound: regorafenib
   MoA     : nerve growth factor receptor trk-a
Found 78 samples for compound 'regorafenib' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 5/5 [00:00<00:00,  6.03it/s]


...

Feature extraction complete.
Text features shape: torch.Size([14, 512])
Image features shape: torch.Size([14, 512])
Cosine Similarity Mean: 0.2515
Cosine Similarity Std:  0.0666

Running matcher for:
   Compound: nilotinib
   MoA     : bcr/abl fusion protein
Found 14 samples for compound 'nilotinib' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 1/1 [00:00<00:00,  5.96it/s]


Feature extraction complete.
Text features shape: torch.Size([14, 512])
Image features shape: torch.Size([14, 512])
Cosine Similarity Mean: 0.3236
Cosine Similarity Std:  0.0854

Running matcher for:
   Compound: oxybuprocaine
   MoA     : sodium channel alpha subunit blocker
Found 15 samples for compound 'oxybuprocaine' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 1/1 [00:00<00:00,  4.94it/s]

Feature extraction complete.
Text features shape: torch.Size([15, 512])
Image features shape: torch.Size([15, 512])
Cosine Similarity Mean: 0.1662
Cosine Similarity Std:  0.0942


In [ ]:
import json
from collections import defaultdict

jsonl_path = "/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl_orig/final/test.jsonl"
compound_moa_counter = defaultdict(int)
compound_to_moas = defaultdict(set)

# 读取 JSONL 文件
with open(jsonl_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        text = data.get("text", "")

        # 提取 compound 和 MOA
        if "The perturbation compound is " in text and "The mechanism of action for this compound is " in text:
            try:
                compound = text.split("The perturbation compound is ")[1].split(";")[0].strip()
                moa = text.split("The mechanism of action for this compound is ")[1].split(".")[0].strip()
                class_key = f"{compound} | {moa}"
                compound_moa_counter[class_key] += 1
                compound_to_moas[compound].add(moa)
            except Exception as e:
                print(f"❌ Error parsing text: {text}")
                print(e)

# 过滤出只对应一个 MOA 的 compound
unique_moa_compound_moa_counter = {}

for class_key, count in compound_moa_counter.items():
    compound, moa = class_key.split(" | ")
    if len(compound_to_moas[compound]) == 1:
        unique_moa_compound_moa_counter[class_key] = count

# 过滤：只保留样本数 > 50 的
filtered = {k: v for k, v in unique_moa_compound_moa_counter.items() if v > 50}

# 输出结果
print("📊 Compound + Mechanism of Action 类别统计（仅显示样本数 > 50 且该药物仅对应一个 MOA 的）：")

if not filtered:
    print("⚠️ 没有符合条件的类别。")
else:
    for class_key, count in sorted(filtered.items(), key=lambda x: -x[1]):
        compound, moa = class_key.split(" | ")
        print(f"🧪 Compound: {compound:15s} | MOA: {moa:35s} → {count} samples")

📊 Compound + Mechanism of Action 类别统计（仅显示样本数 > 50 且该药物仅对应一个 MOA 的）：
🧪 Compound: esomeprazole    | MOA: potassium-transporting atpase inhibitor → 32640 samples
🧪 Compound: regorafenib     | MOA: nerve growth factor receptor trk-a  → 13068 samples
🧪 Compound: sacubitril      | MOA: neprilysin inhibitor                → 13048 samples
🧪 Compound: buparlisib      | MOA: pi3-kinase class i inhibitor        → 10760 samples
🧪 Compound: dexamethasone   | MOA: glucocorticoid receptor agonist     → 10740 samples
🧪 Compound: nimodipine      | MOA: voltage-gated l-type calcium channel blocker → 10380 samples
🧪 Compound: az258           | MOA: aurora kinase inhibitors            → 4464 samples
🧪 Compound: colchicine      | MOA: tubulin inhibitor                   → 4268 samples
🧪 Compound: mg-132          | MOA: protein degradation                 → 4176 samples
🧪 Compound: fluphenazine    | MOA: dopamine d2 receptor antagonist     → 3072 samples
🧪 Compound: dapoxetine      | MOA: serotonin transpor

In [2]:
import json
import numpy as np
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity

def get_matched_samples_by_compound_name(
    jsonl_path: str,
    npz_path: str,
    target_compound: str,
    target_moa: str = None,
    num_samples: int = 100,
    similarity_threshold: float = 0.85,
    seed: int = 42
):
    np.random.seed(seed)

    # 加载 compound 名称和对应的 embedding
    npz_data = np.load(npz_path, allow_pickle=True)
    compound_names_all = list(npz_data['compounds'])
    compound_embeddings_all = npz_data['embeddings']
    name_to_index = {name: idx for idx, name in enumerate(compound_names_all)}

    assert target_compound in name_to_index, f"❌ Compound '{target_compound}' not in compound embedding list."

    target_embedding = compound_embeddings_all[name_to_index[target_compound]].reshape(1, -1)

    matched_samples = []
    with open(jsonl_path, 'r') as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            emb = data.get("compound_embedding")
            if emb is None:
                continue
            emb_np = np.array(emb, dtype=np.float32).reshape(1, -1)

            sim = cosine_similarity(emb_np, target_embedding)[0][0]
            if sim >= similarity_threshold:
                if target_moa:
                    text = data.get("text", "")
                    if f"The mechanism of action for this compound is {target_moa}" not in text:
                        continue
                matched_samples.append(data)
                if len(matched_samples) >= num_samples:
                    break

    print(f"✅ Found {len(matched_samples)} samples for compound '{target_compound}' (sim ≥ {similarity_threshold})")
    return matched_samples

In [7]:
jsonl_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl_moa/final/test_1_cleaned.jsonl'
npz_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl/metadata/all_compounds_embedding.npz'
target_compound = "mg-132"
target_moa = "protein degradation"

matched_samples = get_matched_samples_by_compound_name(
    jsonl_path=jsonl_path,
    npz_path=npz_path,
    target_compound=target_compound,
    target_moa=target_moa,  # 可选
    num_samples=100,
    similarity_threshold=0.4
)

✅ Found 22 samples for compound 'mg-132' (sim ≥ 0.4)


In [8]:
from torch.utils.data import Dataset
import json
import os
import logging
import torch
import numpy as np
from PIL import Image

class JsonlDataset_unseen(Dataset):
    def __init__(self, input_filename=None, samples=None, transforms=None, tokenizer=None, image_base_dir=""):
        assert input_filename or samples, "You must provide either `input_filename` or `samples`."

        if input_filename:
            logging.debug(f'Loading jsonl data from {input_filename}.')
            with open(input_filename, 'r', encoding='utf-8') as f:
                self.samples = [json.loads(line) for line in f]
            logging.debug(f'Loaded {len(self.samples)} samples from file.')
        else:
            self.samples = samples
            logging.debug(f'Loaded {len(self.samples)} samples from memory.')

        self.transforms = transforms
        self.tokenize = tokenizer
        self.image_base_dir = image_base_dir

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image_path = os.path.join(self.image_base_dir, sample['image'])
        full_img = Image.open(image_path).convert("L")  # shape: [H, W] = [512, 1024]
        full_tensor = torch.from_numpy(np.array(full_img)).unsqueeze(0).float() / 255.

        left_tensor = full_tensor[:, :, :512]   # shape: [1, 512, 512]
        right_tensor = full_tensor[:, :, 512:]  # shape: [1, 512, 512]
        image = torch.cat([left_tensor, right_tensor], dim=0)

        if self.tokenize is not None:
            text = self.tokenize([str(sample['text'])])[0]  # tensor shape: [context_length]
        else:
            text = str(sample['text'])

        concentration = torch.tensor(sample["Concentration"], dtype=torch.float32)
        time = torch.tensor(sample["Time"], dtype=torch.float32)
        compound_embedding = torch.tensor(sample["compound_embedding"], dtype=torch.float32)

        return image, text, concentration, time, compound_embedding, str(sample['text'])

In [9]:
image_base_dir = "/ssd2/letitiaz/cp_project/data"
dataset = JsonlDataset_unseen(samples=matched_samples, tokenizer=tokenizer, image_base_dir=image_base_dir)

In [10]:
all_images = []
all_texts = []
all_concentrations = []
all_times = []
all_compound_embeddings = []
all_raw_texts = []

for i in range(len(dataset)):
    image, text, concentration, time, compound_embedding, raw_text = dataset[i]
    all_images.append(image)
    all_texts.append(text)
    all_concentrations.append(concentration)
    all_times.append(time)
    all_compound_embeddings.append(compound_embedding)
    all_raw_texts.append(raw_text)
image_batch = torch.stack(all_images).to(device)                 # [N, 2, 512, 512]
text_batch = torch.stack(all_texts).to(device)                   # [N, context_length
conc_batch = torch.stack(all_concentrations).to(device)         # [N]
time_batch = torch.stack(all_times).to(device)                   # [N]
compound_batch = torch.stack(all_compound_embeddings).to(device) # [N, D]

In [11]:
from tqdm import tqdm
import torch

def compute_features_in_batches(
    model,
    image_batch,
    text_batch,
    conc_batch,
    time_batch,
    compound_batch,
    device,
    batch_size=8
):
    all_image_features = []
    all_text_features = []

    model.eval()
    with torch.no_grad():
        num_samples = len(image_batch)
        num_batches = (num_samples + batch_size - 1) // batch_size

        for i in tqdm(range(0, num_samples, batch_size), desc="Computing features", unit="batch"):
            # 当前 batch 范围
            batch_slice = slice(i, i + batch_size)

            # 将当前 batch 移动到 device
            image = image_batch[batch_slice].to(device)
            text = text_batch[batch_slice].to(device)
            conc = conc_batch[batch_slice].to(device)
            time_ = time_batch[batch_slice].to(device)
            compound = compound_batch[batch_slice].to(device)

            # 编码 text 和 image
            text_feat = model.encode_text(
                text=text,
                concentration=conc,
                time=time_,
                compound_embedding=compound,
                normalize=True
            ).float().cpu()  

            image_feat = model.encode_image(
                image=image,
                normalize=True
            ).float().cpu()  
            # 累加结果
            all_text_features.append(text_feat)
            all_image_features.append(image_feat)

    # 拼接所有小 batch 特征
    all_text_features = torch.cat(all_text_features, dim=0)
    all_image_features = torch.cat(all_image_features, dim=0)

    return all_text_features, all_image_features

text_features, image_features = compute_features_in_batches(
    model=model,
    image_batch=image_batch,
    text_batch=text_batch,
    conc_batch=conc_batch,
    time_batch=time_batch,
    compound_batch=compound_batch,
    device=device,
    batch_size=16
)

print("Text features shape:", text_features.shape)
print("Image features shape:", image_features.shape)

Computing features: 100%|██████████| 2/2 [00:00<00:00,  3.84batch/s]

Text features shape: torch.Size([22, 512])
Image features shape: torch.Size([22, 512])


In [12]:
import torch.nn.functional as F
cos_sim = F.cosine_similarity(text_features, image_features, dim=1)  # [N]

mean_sim = cos_sim.mean().item()
std_sim = cos_sim.std().item()

print("Cosine Similarity Mean:", mean_sim)
print("Cosine Similarity Std:", std_sim)

Cosine Similarity Mean: 0.4482767879962921
Cosine Similarity Std: 0.08052001893520355


In [ ]:
jsonl_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl_moa/final/test_1_cleaned.jsonl'
npz_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl/metadata/all_compounds_embedding.npz'
image_base_dir = "/ssd2/letitiaz/cp_project/data"
target_compound = "az258"
target_moa = "aurora kinase inhibitors"

matcher = CompoundMatcher(
    jsonl_path=jsonl_path,
    npz_path=npz_path,
    image_base_dir=image_base_dir,
    tokenizer=tokenizer,    
    model=model,            
    device = device,
    target_compound=target_compound,
    target_moa=target_moa,
    num_samples=100,
    similarity_threshold=0.4,
    batch_size=16
)

matcher.run_all()

✅ Found 29 samples for compound 'az258' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 2/2 [00:00<00:00,  2.54it/s]

Feature extraction complete.
Text features shape: torch.Size([29, 512])
Image features shape: torch.Size([29, 512])
Cosine Similarity Mean: 0.4675
Cosine Similarity Std:  0.1043


# CLIP-orig

In [1]:
import os
import sys
project_root = "/ssd2/letitiaz/cp_project/code/open-clip/src"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import open_clip
import torch
from test_utiles import *
from unseen_utiles import *

pretrained_ckpt_path = "/ssd2/letitiaz/cp_project/data/logs/ViT-B-16-experiment9/2025_09_07-05_57_02-model_ViT-B-16-lr_0.001-b_175-j_4-p_fp32/checkpoints/epoch_35.pt"
device = torch.device("cuda:7" if torch.cuda.is_available() else "cpu")

tokenizer = open_clip.get_tokenizer(
    model_name="ViT-B-16",
    context_length=256,
    tokenizer_type="hf:gpt2"
)

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    model_name="ViT-B-16",
    pretrained= pretrained_ckpt_path,
    precision="fp32",
    device=device,
    tokenizer=tokenizer,
    output_dict=True,
    load_weights_only=True,
)

model.eval()
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Context length:", context_length)
print("Vocab size:", vocab_size)

/ssd2/letitiaz/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


get_tokenizer called with model_name: ViT-B-16, context_length: 256, tokenizer_type: hf:gpt2
Set pad_token to eos_token for GPT tokenizer
Model parameters: 150,587,905
Context length: 256
Vocab size: 50260


In [6]:
compound_moa_list = [
    ("esomeprazole", "potassium–transporting atpase inhibitor"),
    ("regorafenib", "nerve growth factor receptor trk–a"),
    ("sacubitril", "neprilysin inhibitor"),
    ("buparlisib", "pi3–kinase class i inhibitor"),
    ("dexamethasone", "glucocorticoid receptor agonist"),
    ("nimodipine", "voltage–gated l–type calcium channel blocker"),
    ("az258", "aurora kinase inhibitors"),
    ("colchicine", "tubulin inhibitor"),
    ("mg-132", "protein degradation"),
    ("fluphenazine", "dopamine d2 receptor antagonist"),
    ("dapoxetine", "serotonin transporter inhibitor"),
    ("nilotinib", "bcr/abl fusion protein"),
    ("oxybuprocaine", "sodium channel alpha subunit blocker"),
]

compound_moa_list = [(c, m.replace("–", "-")) for c, m in compound_moa_list]

for compound, moa in compound_moa_list:
    print(f"\nRunning matcher for:")
    print(f"   Compound: {compound}")
    print(f"   MoA     : {moa}")
    print("=" * 60)

    matcher = CompoundMatcher_orig(
        jsonl_path="/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl_orig/final/test.jsonl",
        image_base_dir="/ssd2/letitiaz/cp_project/data",
        tokenizer=tokenizer,
        model=model,
        device=device,
        target_compound=compound,
        target_moa=moa,
        num_samples=100,
        batch_size=16
    )

    try:
        matcher.run_all()
    except Exception as e:
        print(f"❌ Error processing {compound} - {moa}: {e}")


🚀 Running matcher for:
   Compound: esomeprazole
   MoA     : potassium-transporting atpase inhibitor
Found 100 samples with compound 'esomeprazole' and MoA 'potassium-transporting atpase inhibitor'.


Encoding features: 100%|██████████| 7/7 [00:01<00:00,  6.90it/s]


Feature extraction complete.
Text features shape: torch.Size([100, 512])
Image features shape: torch.Size([100, 512])
Cosine Similarity Mean: 0.1834
Cosine Similarity Std:  0.0968

🚀 Running matcher for:
   Compound: regorafenib
   MoA     : nerve growth factor receptor trk-a
Found 100 samples with compound 'regorafenib' and MoA 'nerve growth factor receptor trk-a'.


Encoding features: 100%|██████████| 7/7 [00:01<00:00,  7.00it/s]


Feature extraction complete.
Text features shape: torch.Size([100, 512])
Image features shape: torch.Size([100, 512])
Cosine Similarity Mean: 0.2068
Cosine Similarity Std:  0.0821

🚀 Running matcher for:
   Compound: sacubitril
   MoA     : neprilysin inhibitor
Found 100 samples with compound 'sacubitril' and MoA 'neprilysin inhibitor'.


...

Feature extraction complete.
Text features shape: torch.Size([100, 512])
Image features shape: torch.Size([100, 512])
Cosine Similarity Mean: 0.2305
Cosine Similarity Std:  0.1100

🚀 Running matcher for:
   Compound: nilotinib
   MoA     : bcr/abl fusion protein
Found 100 samples with compound 'nilotinib' and MoA 'bcr/abl fusion protein'.


Encoding features: 100%|██████████| 7/7 [00:01<00:00,  6.39it/s]


Feature extraction complete.
Text features shape: torch.Size([100, 512])
Image features shape: torch.Size([100, 512])
Cosine Similarity Mean: 0.1740
Cosine Similarity Std:  0.0796

🚀 Running matcher for:
   Compound: oxybuprocaine
   MoA     : sodium channel alpha subunit blocker
Found 100 samples with compound 'oxybuprocaine' and MoA 'sodium channel alpha subunit blocker'.


Encoding features: 100%|██████████| 7/7 [00:01<00:00,  6.96it/s]

Feature extraction complete.
Text features shape: torch.Size([100, 512])
Image features shape: torch.Size([100, 512])
Cosine Similarity Mean: 0.1741
Cosine Similarity Std:  0.0935


In [4]:
matcher = CompoundMatcher_orig(
    jsonl_path="/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl_orig/final/test.jsonl",
    image_base_dir="/ssd2/letitiaz/cp_project/data",
    tokenizer=tokenizer,  
    model=model,         
    device=device,
    target_compound = "regorafenib",
    target_moa = "nerve growth factor receptor trk-a",
    num_samples=100,
    batch_size=16
)

matcher.run_all()

Found 100 samples with compound 'regorafenib' and MoA 'nerve growth factor receptor trk-a'.


Loading batches:   0%|          | 0/7 [00:00<?, ?it/s]

Encoding features: 100%|██████████| 7/7 [00:01<00:00,  4.76it/s]

Feature extraction complete.
Text features shape: torch.Size([100, 512])
Image features shape: torch.Size([100, 512])
Cosine Similarity Mean: 0.2068
Cosine Similarity Std:  0.0821


In [3]:
import os
import sys
project_root = "/ssd2/letitiaz/cp_project/code/open-clip/src"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import open_clip
import torch
from test_utiles import *
from unseen_utiles import *

pretrained_ckpt_path = "/ssd2/letitiaz/cp_project/data/logs/ViT-B-16-experiment7/2025_08_31-03_02_38-model_ViT-B-16-lr_0.001-b_175-j_4-p_fp32/checkpoints/epoch_35.pt"
device = torch.device("cuda:7" if torch.cuda.is_available() else "cpu")

tokenizer = open_clip.get_tokenizer(
    model_name="ViT-B-16",
    context_length=256,
    tokenizer_type="hf:gpt2",  
    additional_special_tokens=['<CONC_TOKEN>', '<TIME_TOKEN>', '<COMPOUND_TOKEN>']
)

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    model_name="ViT-B-16",
    pretrained= pretrained_ckpt_path,
    precision="fp32",
    device=device,
    tokenizer=tokenizer,
    output_dict=True,
    load_weights_only=True,
    special_tokens=['<CONC_TOKEN>', '<TIME_TOKEN>', '<COMPOUND_TOKEN>'],
    use_enhanced_clip=True,
    compound_dim=1024
)
model.eval()
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Context length:", context_length)
print("Vocab size:", vocab_size)





get_tokenizer called with model_name: ViT-B-16, context_length: 256, tokenizer_type: hf:gpt2
Added 3 special tokens to gpt2 tokenizer
   <CONC_TOKEN> -> ID: 50257
   <TIME_TOKEN> -> ID: 50258
   <COMPOUND_TOKEN> -> ID: 50259
Set pad_token to eos_token for GPT tokenizer
[Auto] Fetched special token IDs from tokenizer: {'<CONC_TOKEN>': 50257, '<TIME_TOKEN>': 50258, '<COMPOUND_TOKEN>': 50259}
Model parameters: 151,903,233
Context length: 256
Vocab size: 50260


In [6]:
compound_moa_list = [
    ("esomeprazole", "potassium–transporting atpase inhibitor"),
    ("regorafenib", "nerve growth factor receptor trk–a"),
    ("sacubitril", "neprilysin inhibitor"),
    ("buparlisib", "pi3–kinase class i inhibitor"),
    ("dexamethasone", "glucocorticoid receptor agonist"),
    ("nimodipine", "voltage–gated l–type calcium channel blocker"),
    ("az258", "aurora kinase inhibitors"),
    ("colchicine", "tubulin inhibitor"),
    ("mg132", "protein degradation"),
    ("fluphenazine", "dopamine d2 receptor antagonist"),
    ("dapoxetine", "serotonin transporter inhibitor"),
    ("nilotinib", "bcr/abl fusion protein"),
    ("oxybuprocaine", "sodium channel alpha subunit blocker"),
]

compound_moa_list = [(c, m.replace("–", "-")) for c, m in compound_moa_list]
jsonl_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl_fp/final/test.jsonl'
npz_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl/metadata/compoundFingerprint.npz'
image_base_dir = "/ssd2/letitiaz/cp_project/data"
for compound, moa in compound_moa_list:
    print(f"\nRunning matcher for:")
    print(f"   Compound: {compound}")
    print(f"   MoA     : {moa}")
    print("=" * 60)

    matcher = CompoundMatcher(
        jsonl_path=jsonl_path,
        npz_path=npz_path,
        image_base_dir=image_base_dir,
        tokenizer=tokenizer,
        model=model,
        device=device,
        target_compound=compound,
        target_moa=moa,
        num_samples=100,
        similarity_threshold=0.4,  
        batch_size=16
    )

    matcher.run_all()



Running matcher for:
   Compound: esomeprazole
   MoA     : potassium-transporting atpase inhibitor
Found 100 samples for compound 'esomeprazole' (sim ≥ 0.4)


Loading batches:   0%|          | 0/7 [00:00<?, ?it/s]

Encoding batches: 100%|██████████| 7/7 [00:01<00:00,  6.01it/s]


Feature extraction complete.
Text features shape: torch.Size([100, 512])
Image features shape: torch.Size([100, 512])
Cosine Similarity Mean: 0.2156
Cosine Similarity Std:  0.1037

Running matcher for:
   Compound: regorafenib
   MoA     : nerve growth factor receptor trk-a
Found 100 samples for compound 'regorafenib' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 7/7 [00:00<00:00,  8.35it/s]


...

Feature extraction complete.
Text features shape: torch.Size([100, 512])
Image features shape: torch.Size([100, 512])
Cosine Similarity Mean: 0.1953
Cosine Similarity Std:  0.0902

Running matcher for:
   Compound: nilotinib
   MoA     : bcr/abl fusion protein
Found 100 samples for compound 'nilotinib' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 7/7 [00:01<00:00,  6.62it/s]


Feature extraction complete.
Text features shape: torch.Size([100, 512])
Image features shape: torch.Size([100, 512])
Cosine Similarity Mean: 0.2101
Cosine Similarity Std:  0.1094

Running matcher for:
   Compound: oxybuprocaine
   MoA     : sodium channel alpha subunit blocker
Found 100 samples for compound 'oxybuprocaine' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 7/7 [00:01<00:00,  5.72it/s]

Feature extraction complete.
Text features shape: torch.Size([100, 512])
Image features shape: torch.Size([100, 512])
Cosine Similarity Mean: 0.1506
Cosine Similarity Std:  0.1014


ViT-L

In [7]:
import os
import sys
project_root = "/ssd2/letitiaz/cp_project/code/open-clip/src"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import open_clip
import torch
from test_utiles import *

pretrained_ckpt_path = "/ssd2/letitiaz/cp_project/data/logs/ViT-L-16-experiment10/2025_09_08-14_51_56-model_ViT-L-16-lr_0.001-b_68-j_4-p_fp32/checkpoints/epoch_30.pt"
device = torch.device("cuda:7" if torch.cuda.is_available() else "cpu")

tokenizer = open_clip.get_tokenizer(
    model_name="ViT-L-16",
    context_length=256,
    tokenizer_type="hf:gpt2",  
    additional_special_tokens=['<CONC_TOKEN>', '<TIME_TOKEN>', '<COMPOUND_TOKEN>']
)

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    model_name="ViT-L-16",
    pretrained= pretrained_ckpt_path,
    precision="fp32",
    device=device,
    tokenizer=tokenizer,
    output_dict=True,
    load_weights_only=True,
    special_tokens=['<CONC_TOKEN>', '<TIME_TOKEN>', '<COMPOUND_TOKEN>'],
    use_enhanced_clip=True  
)
model.eval()
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Context length:", context_length)
print("Vocab size:", vocab_size)





get_tokenizer called with model_name: ViT-L-16, context_length: 256, tokenizer_type: hf:gpt2
Added 3 special tokens to gpt2 tokenizer
   <CONC_TOKEN> -> ID: 50257
   <TIME_TOKEN> -> ID: 50258
   <COMPOUND_TOKEN> -> ID: 50259
Set pad_token to eos_token for GPT tokenizer
[Auto] Fetched special token IDs from tokenizer: {'<CONC_TOKEN>': 50257, '<TIME_TOKEN>': 50258, '<COMPOUND_TOKEN>': 50259}
Model parameters: 431,026,945
Context length: 256
Vocab size: 50260


In [9]:
compound_moa_list = [
    ("esomeprazole", "potassium–transporting atpase inhibitor"),
    ("regorafenib", "nerve growth factor receptor trk–a"),
    ("sacubitril", "neprilysin inhibitor"),
    ("buparlisib", "pi3–kinase class i inhibitor"),
    ("dexamethasone", "glucocorticoid receptor agonist"),
    ("nimodipine", "voltage–gated l–type calcium channel blocker"),
    ("az258", "aurora kinase inhibitors"),
    ("colchicine", "tubulin inhibitor"),
    ("mg132", "protein degradation"),
    ("fluphenazine", "dopamine d2 receptor antagonist"),
    ("dapoxetine", "serotonin transporter inhibitor"),
    ("nilotinib", "bcr/abl fusion protein"),
    ("oxybuprocaine", "sodium channel alpha subunit blocker"),
]

compound_moa_list = [(c, m.replace("–", "-")) for c, m in compound_moa_list]
jsonl_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl_moa/final/test_1_cleaned.jsonl'
npz_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl/metadata/all_compounds_embedding.npz'
image_base_dir = "/ssd2/letitiaz/cp_project/data"
for compound, moa in compound_moa_list:
    print(f"\nRunning matcher for:")
    print(f"   Compound: {compound}")
    print(f"   MoA     : {moa}")
    print("=" * 60)

    matcher = CompoundMatcher(
        jsonl_path=jsonl_path,
        npz_path=npz_path,
        image_base_dir=image_base_dir,
        tokenizer=tokenizer,
        model=model,
        device=device,
        target_compound=compound,
        target_moa=moa,
        num_samples=100,
        similarity_threshold=0.4,  
        batch_size=16
    )

    matcher.run_all()



Running matcher for:
   Compound: esomeprazole
   MoA     : potassium-transporting atpase inhibitor
Found 100 samples for compound 'esomeprazole' (sim ≥ 0.4)


Loading batches:   0%|          | 0/7 [00:00<?, ?it/s]

Encoding batches: 100%|██████████| 7/7 [00:02<00:00,  2.63it/s]


Feature extraction complete.
Text features shape: torch.Size([100, 768])
Image features shape: torch.Size([100, 768])
Cosine Similarity Mean: 0.2382
Cosine Similarity Std:  0.0946

Running matcher for:
   Compound: regorafenib
   MoA     : nerve growth factor receptor trk-a
Found 78 samples for compound 'regorafenib' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 5/5 [00:02<00:00,  2.38it/s]


...

Feature extraction complete.
Text features shape: torch.Size([14, 768])
Image features shape: torch.Size([14, 768])
Cosine Similarity Mean: 0.2530
Cosine Similarity Std:  0.1067

Running matcher for:
   Compound: nilotinib
   MoA     : bcr/abl fusion protein
Found 14 samples for compound 'nilotinib' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 1/1 [00:00<00:00,  2.30it/s]


Feature extraction complete.
Text features shape: torch.Size([14, 768])
Image features shape: torch.Size([14, 768])
Cosine Similarity Mean: 0.2947
Cosine Similarity Std:  0.0890

Running matcher for:
   Compound: oxybuprocaine
   MoA     : sodium channel alpha subunit blocker
Found 15 samples for compound 'oxybuprocaine' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 1/1 [00:00<00:00,  2.72it/s]

Feature extraction complete.
Text features shape: torch.Size([15, 768])
Image features shape: torch.Size([15, 768])
Cosine Similarity Mean: 0.2105
Cosine Similarity Std:  0.0854


# SigLIP descriptor

In [4]:
import os
import sys
project_root = "/ssd2/letitiaz/cp_project/code/open-clip/src"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import open_clip
import torch
from test_utiles import *
from unseen_utiles import *

pretrained_ckpt_path = "/ssd2/letitiaz/cp_project/data/logs/ViT-B-16-experiment11/2025_09_18-01_45_57-model_ViT-B-16-SigLIP-512-lr_0.001-b_170-j_4-p_fp32/checkpoints/epoch_35.pt"
device = torch.device("cuda:7" if torch.cuda.is_available() else "cpu")

tokenizer = open_clip.get_tokenizer(
    model_name="ViT-B-16-SigLIP-512",
    context_length=256,
    tokenizer_type="hf:gpt2",  
    additional_special_tokens=['<CONC_TOKEN>', '<TIME_TOKEN>', '<COMPOUND_TOKEN>']
)

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    model_name="ViT-B-16-SigLIP-512",
    pretrained= pretrained_ckpt_path,
    precision="fp32",
    device=device,
    tokenizer=tokenizer,
    output_dict=True,
    load_weights_only=True,
    special_tokens=['<CONC_TOKEN>', '<TIME_TOKEN>', '<COMPOUND_TOKEN>'],
    use_enhanced_clip=True,
)
model.eval()
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Context length:", context_length)
print("Vocab size:", vocab_size)





get_tokenizer called with model_name: ViT-B-16-SigLIP-512, context_length: 256, tokenizer_type: hf:gpt2
Added 3 special tokens to gpt2 tokenizer
   <CONC_TOKEN> -> ID: 50257
   <TIME_TOKEN> -> ID: 50258
   <COMPOUND_TOKEN> -> ID: 50259
Set pad_token to eos_token for GPT tokenizer
[Auto] Fetched special token IDs from tokenizer: {'<CONC_TOKEN>': 50257, '<TIME_TOKEN>': 50258, '<COMPOUND_TOKEN>': 50259}
Model parameters: 219,676,418
Context length: 256
Vocab size: 50260


In [6]:
compound_moa_list = [
    ("esomeprazole", "potassium–transporting atpase inhibitor"),
    ("regorafenib", "nerve growth factor receptor trk–a"),
    ("sacubitril", "neprilysin inhibitor"),
    ("buparlisib", "pi3–kinase class i inhibitor"),
    ("dexamethasone", "glucocorticoid receptor agonist"),
    ("nimodipine", "voltage–gated l–type calcium channel blocker"),
    ("az258", "aurora kinase inhibitors"),
    ("colchicine", "tubulin inhibitor"),
    ("mg132", "protein degradation"),
    ("fluphenazine", "dopamine d2 receptor antagonist"),
    ("dapoxetine", "serotonin transporter inhibitor"),
    ("nilotinib", "bcr/abl fusion protein"),
    ("oxybuprocaine", "sodium channel alpha subunit blocker"),
]

compound_moa_list = [(c, m.replace("–", "-")) for c, m in compound_moa_list]
jsonl_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl_moa/final/test_1_cleaned.jsonl'
npz_path = '/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl/metadata/all_compounds_embedding.npz'
image_base_dir = "/ssd2/letitiaz/cp_project/data"
for compound, moa in compound_moa_list:
    print(f"\nRunning matcher for:")
    print(f"   Compound: {compound}")
    print(f"   MoA     : {moa}")
    print("=" * 60)

    matcher = CompoundMatcher(
        jsonl_path=jsonl_path,
        npz_path=npz_path,
        image_base_dir=image_base_dir,
        tokenizer=tokenizer,
        model=model,
        device=device,
        target_compound=compound,
        target_moa=moa,
        num_samples=100,
        similarity_threshold=0.4,  
        batch_size=16
    )

    matcher.run_all()



Running matcher for:
   Compound: esomeprazole
   MoA     : potassium-transporting atpase inhibitor
Found 100 samples for compound 'esomeprazole' (sim ≥ 0.4)


Loading batches:   0%|          | 0/7 [00:00<?, ?it/s]

Encoding batches: 100%|██████████| 7/7 [00:01<00:00,  5.89it/s]


Feature extraction complete.
Text features shape: torch.Size([100, 768])
Image features shape: torch.Size([100, 768])
Cosine Similarity Mean: 0.2963
Cosine Similarity Std:  0.1162

Running matcher for:
   Compound: regorafenib
   MoA     : nerve growth factor receptor trk-a
Found 78 samples for compound 'regorafenib' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 5/5 [00:00<00:00,  5.34it/s]


...

Feature extraction complete.
Text features shape: torch.Size([14, 768])
Image features shape: torch.Size([14, 768])
Cosine Similarity Mean: 0.3591
Cosine Similarity Std:  0.0458

Running matcher for:
   Compound: nilotinib
   MoA     : bcr/abl fusion protein
Found 14 samples for compound 'nilotinib' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 1/1 [00:00<00:00,  6.64it/s]


Feature extraction complete.
Text features shape: torch.Size([14, 768])
Image features shape: torch.Size([14, 768])
Cosine Similarity Mean: 0.3034
Cosine Similarity Std:  0.0901

Running matcher for:
   Compound: oxybuprocaine
   MoA     : sodium channel alpha subunit blocker
Found 15 samples for compound 'oxybuprocaine' (sim ≥ 0.4)


Encoding batches: 100%|██████████| 1/1 [00:00<00:00,  5.15it/s]

Feature extraction complete.
Text features shape: torch.Size([15, 768])
Image features shape: torch.Size([15, 768])
Cosine Similarity Mean: 0.2667
Cosine Similarity Std:  0.1111


# SigLIP orig

In [1]:
import os
import sys
project_root = "/ssd2/letitiaz/cp_project/code/open-clip/src"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import open_clip
import torch
from test_utiles import *
from unseen_utiles import *

pretrained_ckpt_path = "/ssd2/letitiaz/cp_project/data/logs/ViT-B-16-experiment12/2025_09_22-13_14_15-model_ViT-B-16-SigLIP-512-lr_0.001-b_170-j_4-p_fp32/checkpoints/epoch_35.pt"
device = torch.device("cuda:7" if torch.cuda.is_available() else "cpu")

tokenizer = open_clip.get_tokenizer(
    model_name="ViT-B-16-SigLIP-512",
    context_length=256,
    tokenizer_type="hf:gpt2",  
)

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    model_name="ViT-B-16-SigLIP-512",
    pretrained= pretrained_ckpt_path,
    precision="fp32",
    device=device,
    tokenizer=tokenizer,
    output_dict=True,
    load_weights_only=True,
)
model.eval()
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Context length:", context_length)
print("Vocab size:", vocab_size)





/ssd2/letitiaz/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


get_tokenizer called with model_name: ViT-B-16-SigLIP-512, context_length: 256, tokenizer_type: hf:gpt2
Set pad_token to eos_token for GPT tokenizer
Model parameters: 217,766,402
Context length: 256
Vocab size: 50260


In [2]:
compound_moa_list = [
    ("esomeprazole", "potassium–transporting atpase inhibitor"),
    ("regorafenib", "nerve growth factor receptor trk–a"),
    ("sacubitril", "neprilysin inhibitor"),
    ("buparlisib", "pi3–kinase class i inhibitor"),
    ("dexamethasone", "glucocorticoid receptor agonist"),
    ("nimodipine", "voltage–gated l–type calcium channel blocker"),
    ("az258", "aurora kinase inhibitors"),
    ("colchicine", "tubulin inhibitor"),
    ("mg-132", "protein degradation"),
    ("fluphenazine", "dopamine d2 receptor antagonist"),
    ("dapoxetine", "serotonin transporter inhibitor"),
    ("nilotinib", "bcr/abl fusion protein"),
    ("oxybuprocaine", "sodium channel alpha subunit blocker"),
]

compound_moa_list = [(c, m.replace("–", "-")) for c, m in compound_moa_list]

for compound, moa in compound_moa_list:
    print(f"\nRunning matcher for:")
    print(f"   Compound: {compound}")
    print(f"   MoA     : {moa}")
    print("=" * 60)

    matcher = CompoundMatcher_orig(
        jsonl_path="/ssd2/letitiaz/cp_project/data/jsonFile/experimentJsonl_orig/final/test.jsonl",
        image_base_dir="/ssd2/letitiaz/cp_project/data",
        tokenizer=tokenizer,
        model=model,
        device=device,
        target_compound=compound,
        target_moa=moa,
        num_samples=100,
        batch_size=16
    )

    try:
        matcher.run_all()
    except Exception as e:
        print(f"❌ Error processing {compound} - {moa}: {e}")


Running matcher for:
   Compound: esomeprazole
   MoA     : potassium-transporting atpase inhibitor
Found 100 samples with compound 'esomeprazole' and MoA 'potassium-transporting atpase inhibitor'.


Encoding features: 100%|██████████| 7/7 [00:01<00:00,  6.16it/s]


Feature extraction complete.
Text features shape: torch.Size([100, 768])
Image features shape: torch.Size([100, 768])
Cosine Similarity Mean: 0.0072
Cosine Similarity Std:  0.1175

Running matcher for:
   Compound: regorafenib
   MoA     : nerve growth factor receptor trk-a
Found 100 samples with compound 'regorafenib' and MoA 'nerve growth factor receptor trk-a'.


Encoding features: 100%|██████████| 7/7 [00:00<00:00, 10.82it/s]


Feature extraction complete.
Text features shape: torch.Size([100, 768])
Image features shape: torch.Size([100, 768])
Cosine Similarity Mean: 0.0381
Cosine Similarity Std:  0.0817

Running matcher for:
   Compound: sacubitril
   MoA     : neprilysin inhibitor
Found 100 samples with compound 'sacubitril' and MoA 'neprilysin inhibitor'.


...

Feature extraction complete.
Text features shape: torch.Size([100, 768])
Image features shape: torch.Size([100, 768])
Cosine Similarity Mean: -0.0367
Cosine Similarity Std:  0.0972

Running matcher for:
   Compound: nilotinib
   MoA     : bcr/abl fusion protein
Found 100 samples with compound 'nilotinib' and MoA 'bcr/abl fusion protein'.


Encoding features: 100%|██████████| 7/7 [00:00<00:00,  9.59it/s]


Feature extraction complete.
Text features shape: torch.Size([100, 768])
Image features shape: torch.Size([100, 768])
Cosine Similarity Mean: -0.0549
Cosine Similarity Std:  0.1035

Running matcher for:
   Compound: oxybuprocaine
   MoA     : sodium channel alpha subunit blocker
Found 100 samples with compound 'oxybuprocaine' and MoA 'sodium channel alpha subunit blocker'.


Encoding features: 100%|██████████| 7/7 [00:00<00:00, 10.32it/s]

Feature extraction complete.
Text features shape: torch.Size([100, 768])
Image features shape: torch.Size([100, 768])
Cosine Similarity Mean: -0.0323
Cosine Similarity Std:  0.0909
